# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [15]:
print(y_train_tensor.shape)

torch.Size([20000, 1])


In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [16]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 19501.461, Val Loss: 19748.541


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 5293.116, Val Loss: 17474.867


In [17]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [18]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$46 $53 $13 $12 $44 $183 $36 $72 $28 $25 $456 $134 $87 $158 $11 $23 $1 $16 $83 $39 $32 $19 $57 $28 $280 $265 $228 $26 $8 $38 $39 $157 $17 $31 $104 $280 $37 $138 $133 $63 $157 $76 $12 $93 $118 $43 $77 $56 $19 $23 $8 $55 $119 $26 $127 $63 $35 $148 $22 $37 $67 $4 $41 $10 $413 $158 $7 $269 $6 $220 $7 $26 $119 $110 $11 $50 $86 $43 $25 $62 $70 $128 $24 $44 $13 $52 $50 $149 $110 $133 $22 $97 $22 $12 $34 $88 $43 $39 $102 $227 $8 $65 $3 $46 $15 $64 $97 $220 $10 $108 $17 $92 $146 $39 $12 $131 $135 $43 $62 $38 $26 $208 $19 $2 $85 $17 $22 $143 $57 $53 $41 $140 $133 $26 $64 $28 $113 $68 $42 $75 $18 $161 $5 $129 $201 $73 $42 $280 $90 $10 $21 $214 $5 $75 $40 $142 $153 $12 $4 $7 $80 $12 $3 $27 $517 $16 $60 $12 $17 $40 $16 $21 $256 $48 $15 $0 $23 $8 $32 $129 $383 $17 $94 $10 $28 $90 $41 $61 $13 $15 $37 $72 $59 $36 $14 $23 $59 $22 $7 $16 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [19]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [20]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [21]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [26]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="ollama/gemma3:4b", messages=messages_for(item), api_base="http://localhost:11434")
    return response.choices[0].message.content

In [27]:
gpt_4__1_nano(test[0])

'$349'

In [28]:
test[0].price

219.0

In [29]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$130 $14 $25 $34 $0 $140 $104 $85 $10 $20 $214 $20 $10 $19 $1 $20 $11 $30 $40 $31 $99 $16 $5 $225 $152 $274 $205 $15 $250 $65 $330 $20 $90 $54 $25 $20 $190 $26 $36 $19 $160 $50 $15 $55 $140 $5 $18 $8 $65 $112 $16 $105 $76 $20 $137 $44 $6 $30 $22 $7 $106 $38 $58 $24 $529 $10 $150 $355 $25 $104 $19 $3 $120 $17 $30 $21 $226 $5 $8 $6 $20 $3 $40 $74 $7 $10 $218 $93 $50 $26 $2 $45 $1 $10 $1 $98 $6 $242 $200 $345 $30 $27 $12 $49 $219 $232 $12 $250 $5 $51 $100 $36 $59 $38 $74 $529 $0 $5 $74 $47 $9 $410 $0 $74 $150 $80 $15 $9 $29 $103 $108 $7 $19 $0 $125 $10 $125 $30 $100 $52 $26 $249 $40 $11 $104 $3 $20 $390 $15 $13 $4 $184 $3 $159 $11 $149 $41 $47 $30 $30 $0 $17 $23 $0 $41 $2 $48 $24 $0 $15 $0 $3 $80 $3 $37 $201 $2 $87 $24 $58 $145 $15 $150 $49 $90 $8 $63 $2 $20 $14 $5 $19 $5 $561 $60 $70 $20 $130 $21 $11 

In [30]:
def claude_opus_4_5(item):
    response = completion(model="ollama/llama3.2:3b", messages=messages_for(item), api_base="http://localhost:11434")
    return response.choices[0].message.content

In [31]:
claude_opus_4_5(test[0])

'$169'

In [32]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$70 $46 $35 $79 $20 $205 $94 $95 $14 $520 $588 $420 $120 $4 $19 $18 $9 $8 $140 $1 $35 $44 $40 $35 $207 $284 $1004 $5 $250 $40 $230 $10 $10 $55 $35 $320 $80 $6 $54 $2 $175 $55 $15 $125 $120 $10 $3 $7 $47 $147 $16 $80 $326 $39 $946 $75 $12 $205 $52 $30 $106 $72 $46 $29 $539 $39 $10 $200 $0 $67 $16 $8 $115 $26 $40 $20 $574 $20 $8 $7 $75 $13 $15 $79 $2 $40 $122 $144 $50 $53 $28 $45 $5 $35 $6 $29 $16 $27 $135 $180 $50 $91 $19 $60 $199 $477 $10 $375 $23 $69 $9 $181 $268 $12 $16 $1029 $1 $5 $6 $47 $29 $306 $30 $141 $158 $5 $5 $201 $20 $114 $59 $8 $15 $10 $0 $10 $75 $80 $109 $92 $24 $45 $15 $0 $114 $8 $20 $60 $15 $8 $1 $124 $22 $109 $6 $199 $37 $41 $85 $30 $210 $19 $17 $0 $259 $27 $19 $35 $30 $2 $10 $23 $175 $17 $62 $250 $72 $23 $49 $33 $305 $25 $749 $99 $75 $8 $83 $62 $45 $2 $115 $2 $15 $260 $150 $50 $29 $35 $1 $11 

In [37]:
def gemini_3_pro_preview(item):
    response = completion(model="ollama/phi4-mini:3.8b", messages=messages_for(item), api_base="http://localhost:11434")
    return response.choices[0].message.content

In [38]:
gemini_3_pro_preview(test[0])

'$129.99'

In [39]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$69 $183 $5 $30 $5 $180 $114 $135 $44 $870 $263 $821 $280 $9 $21 $7 $21 $80 $60 $11 $4 $256 $5 $125 $252 $3 $355 $0 $101 $60 $930 $30 $10 $10 $335 $681 $240 $41 $124 $8 $185 $50 $19 $135 $120 $5 $67 $3 $65 $42 

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)